<a href="https://colab.research.google.com/github/Sanchit-hub/UCS547-Accelerated-Data-Science/blob/main/UCS547_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Q1. Identify !, %, and %% used in cell in Google Colab.

In [1]:
# Example of %% command
#-- Writes cell contet into a file named hello.cu
%%writefile helllo.cu
#include <stdio.h>
__global__ void hello() {
    printf("Hello GPU\n");
}


Writing helllo.cu


In [2]:
#Examples of !


!pwd #--Shows present working directory




/content


In [3]:
!ls #--Lists files in the current directory

helllo.cu  sample_data


In [4]:
#Examples for %
%cd /content






/content


In [5]:
%time x = sum(range(10_000)) #-- Measures execution time for the line


CPU times: user 276 µs, sys: 0 ns, total: 276 µs
Wall time: 280 µs


##Q2. Identify all key nvidia-smi commands with multiple options

In [6]:
!nvidia-smi
#This displays overall GPU status and usage information

Fri Mar  6 10:06:10 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
!nvidia-smi -L
#-----to check how many gpus are present ---------------
#Lists all available gpus

GPU 0: Tesla T4 (UUID: GPU-e788504b-26bc-a4e7-555e-89a69f687b69)


In [8]:
!nvidia-smi -q
#---Shows: Detailed gpu usage. For example: Temperature, power usage, memory details, clock speed etc




==============NVSMI LOG==============

Timestamp                                 : Fri Mar  6 10:06:11 2026
Driver Version                            : 580.82.07
CUDA Version                              : 13.0

Attached GPUs                             : 1
GPU 00000000:00:04.0
    Product Name                          : Tesla T4
    Product Brand                         : NVIDIA
    Product Architecture                  : Turing
    Display Mode                          : Requested functionality has been deprecated
    Display Attached                      : Yes
    Display Active                        : Disabled
    Persistence Mode                      : Disabled
    Addressing Mode                       : None
    MIG Mode
        Current                           : N/A
        Pending                           : N/A
    Accounting Mode                       : Disabled
    Accounting Mode Buffer Size           : 4000
    Driver Model
        Current                           : N/

In [9]:
!nvidia-smi --help
# shows all supported options and flags

NVIDIA System Management Interface -- v580.82.07

NVSMI provides monitoring information for Tesla and select Quadro devices.
The data is presented in either a plain text or an XML format, via stdout or a file.
NVSMI also provides several management operations for changing the device state.

Note that the functionality of NVSMI is exposed through the NVML C-based
library. See the NVIDIA developer website for more information about NVML.
Python wrappers to NVML are also available.  The output of NVSMI is
not guaranteed to be backwards compatible; NVML and the bindings are backwards
compatible.

http://developer.nvidia.com/nvidia-management-library-nvml/
http://pypi.python.org/pypi/nvidia-ml-py/
Supported products:
- Full Support
    - All Tesla products, starting with the Kepler architecture
    - All Quadro products, starting with the Kepler architecture
    - All GRID products, starting with the Kepler architecture
    - GeForce Titan products, starting with the Kepler architecture
- L

In [10]:
!nvidia-smi -h

NVIDIA System Management Interface -- v580.82.07

NVSMI provides monitoring information for Tesla and select Quadro devices.
The data is presented in either a plain text or an XML format, via stdout or a file.
NVSMI also provides several management operations for changing the device state.

Note that the functionality of NVSMI is exposed through the NVML C-based
library. See the NVIDIA developer website for more information about NVML.
Python wrappers to NVML are also available.  The output of NVSMI is
not guaranteed to be backwards compatible; NVML and the bindings are backwards
compatible.

http://developer.nvidia.com/nvidia-management-library-nvml/
http://pypi.python.org/pypi/nvidia-ml-py/
Supported products:
- Full Support
    - All Tesla products, starting with the Kepler architecture
    - All Quadro products, starting with the Kepler architecture
    - All GRID products, starting with the Kepler architecture
    - GeForce Titan products, starting with the Kepler architecture
- L

In [11]:
!nvidia-smi -l 1 #Updates GPU statistics every 1 second, used for real-time monitoring



Streaming output truncated to the last 5000 lines.
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+------------------------+----------------------+

+-----------------------------------------------------------------------------------------+
| Processes:                                                                              |
|  GPU   GI   CI              PID   Type   Process name                        GPU Memory |
|        ID   ID            

##Q3.  Debug common CUDA errors (zero output, incorrect indexing, PTX errors)

In [12]:
%%writefile zero_fix.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void helloKernel() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}

int main() {
    helloKernel<<<1, 4>>>();
    cudaDeviceSynchronize();   // Fix: wait for GPU-- if not there then GPU output may not appear because the program ends before the kernel finishes.

    printf("Hello from CPU\n");
    return 0;
}

Writing zero_fix.cu


In [13]:
!nvcc -arch=sm_75 zero_fix.cu -o zero_fix && ./zero_fix

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from CPU


Incorrect Indexing Error
Problem

Using only threadIdx.x ignores block number → wrong results for multiple blocks.

In [14]:
%%writefile index_fix.cu
#include <stdio.h>
#include <cuda_runtime.h>

__global__ void printID() {
    int id = blockIdx.x * blockDim.x + threadIdx.x; //  Wrong indexing-->int id = threadIdx.x;
    printf("Thread id = %d\n", id);
}

int main() {
    printID<<<2, 4>>>();   // 8 total threads
    cudaDeviceSynchronize();
    return 0;
}

Writing index_fix.cu


 PTX / Architecture Error
Problem

Compiled for wrong GPU architecture.

In [15]:
%%writefile ptx_demo.cu
#include <stdio.h>

__global__ void testKernel() {
    printf("GPU running correctly\n");
}

int main() {
    testKernel<<<1,1>>>();
    cudaDeviceSynchronize();
    return 0;
}

Writing ptx_demo.cu


In [16]:
!nvcc -arch=sm_30 ptx_demo.cu -o ptx_demo ./ptx.demo //wrong compile --causes ptx error

nvcc fatal   : Unknown option '--causes'


In [17]:
!nvcc -arch=sm_75 ptx_demo.cu -o ptx_demo ./ptx_demo  //correct compile

nvcc fatal   : Don't know what to do with 'compile'


##Q4. Write a CUDA C/C++ program to demonstrate GPU kernel execution and thread indexing.
a. Launch a CUDA kernel using: 1 block and 8 threads

b. Each thread must print: Hello from GPU thread <global_thread_id>

c. Compute the global thread ID using: global_thread_id = blockIdx.x * blockDim.x + threadIdx.x

d. Clearly separate: Host code (CPU) & Device code (GPU kernel)

In [18]:
%%writefile answer.cu
#include <stdio.h>
#include <cuda_runtime.h>

// ---------------Device code (GPU kernel)-------------------
__global__ void helloKernel() {
    int global_thread_id = blockIdx.x * blockDim.x + threadIdx.x;
    printf("Hello from GPU thread %d\n", global_thread_id);
}

// ------------- Host code (CPU)---------------------------
int main() {
    helloKernel<<<1, 8>>>();
    cudaDeviceSynchronize();

    printf("Hello from CPU\n");
    return 0;
}



Writing answer.cu


In [19]:
!nvcc -arch=sm_75 answer.cu -o answer && ./answer

Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3
Hello from GPU thread 4
Hello from GPU thread 5
Hello from GPU thread 6
Hello from GPU thread 7
Hello from CPU


##Q5. Write a CUDA program to demonstrate host and device memory separation.
a. Create an integer array of size 5 on the host (CPU).

b. Allocate corresponding memory on the device (GPU) using cudaMalloc().

c. Copy data from host to device using cudaMemcpy().

d. Launch a kernel where GPU threads print values from device memory.

e. Copy the data back from device to host and print it on CPU.

In [20]:
%%writefile mem_separation.cu
#include <stdio.h>
#include <cuda_runtime.h>

// ---------- Device code (GPU) ----------
__global__ void show(int *d) {
    int i = threadIdx.x;              // 1 block only, so threadIdx.x is enough
    printf("GPU: d[%d] = %d\n", i, d[i]);
}

// ---------- Host code (CPU) ----------
int main() {
    int h[5] = {10, 20, 30, 40, 50};
    int *d;

    cudaMalloc(&d, 5 * sizeof(int));
    cudaMemcpy(d, h, 5 * sizeof(int), cudaMemcpyHostToDevice);

    show<<<1, 5>>>(d);
    cudaDeviceSynchronize();

    cudaMemcpy(h, d, 5 * sizeof(int), cudaMemcpyDeviceToHost);

    printf("\nCPU after copy back:\n");
    for (int i = 0; i < 5; i++) {
        printf("CPU: h[%d] = %d\n", i, h[i]);
    }

    cudaFree(d);
    return 0;
}

Writing mem_separation.cu


In [21]:
!nvcc -arch=sm_75 mem_separation.cu -o mem_separation

In [22]:
!./mem_separation

GPU: d[0] = 10
GPU: d[1] = 20
GPU: d[2] = 30
GPU: d[3] = 40
GPU: d[4] = 50

CPU after copy back:
CPU: h[0] = 10
CPU: h[1] = 20
CPU: h[2] = 30
CPU: h[3] = 40
CPU: h[4] = 50


##Q6. Compare CPU times of List/tuple with Numpy arrays.

In [23]:
import time
import numpy as np

N = 5_000_000

lst = list(range(N))
tup = tuple(range(N))
arr = np.arange(N)

# ---- 1) Sum comparison ----
start = time.time()
s1 = sum(lst)
t_list_sum = time.time() - start

start = time.time()
s2 = sum(tup)
t_tuple_sum = time.time() - start

start = time.time()
s3 = np.sum(arr)
t_numpy_sum = time.time() - start

print("SUM timings (seconds)")
print("List :", t_list_sum)
print("Tuple:", t_tuple_sum)
print("NumPy:", t_numpy_sum)

# ---- 2) Element-wise operation: x*2 ----
start = time.time()
lst2 = [x * 2 for x in lst]
t_list_op = time.time() - start

start = time.time()
arr2 = arr * 2
t_numpy_op = time.time() - start

print("\nELEMENT-WISE x*2 timings (seconds)")
print("List  :", t_list_op)
print("NumPy :", t_numpy_op)

SUM timings (seconds)
List : 0.06816411018371582
Tuple: 0.0670781135559082
NumPy: 0.0038976669311523438

ELEMENT-WISE x*2 timings (seconds)
List  : 0.36058974266052246
NumPy : 0.015807628631591797
